# Results Loader for all Models

In [1]:
from pathlib import Path

import pandas as pd

from config.config import Config
from src.data import time_series_split
from src.utils import load_results_from_dir, results_to_df, set_seed

In [2]:
cfg = Config.load()
SEED = cfg.runtime.seed
HORIZON = cfg.runtime.horizon
TARGET_MODE = cfg.runtime.target_mode
SAVE = True
rng = set_seed(SEED)

OUT_DIR = Path(cfg.data.processed_dir)
FIG_DIR = Path(cfg.data.fig_dir)

2025-10-26 18:53:00,312 - INFO - src.utils - Global random seed set to 42


In [3]:
results = load_results_from_dir(OUT_DIR)
len(results), [r["kind"] for r in results]

(10,
 ['ensemble',
  'ensemble_wo_sent',
  'linreg',
  'linreg_wo_sent',
  'lstm',
  'lstm_wo_sent',
  'random_forest',
  'random_forest_wo_sent',
  'xgboost',
  'xgboost_wo_sent'])

In [4]:
keep = {"linreg", "linreg_wo_sent"}
results_par = [r for r in results if r["kind"] in keep]

In [5]:
params_summary = results_to_df(results_par, "best_params")
params_summary

,model,alpha,l1_ratio,penalty,max_iter,learning_rate,eta0,random_state
0,linreg,0.01,0.832443,l1,2000,constant,0.01,42
1,linreg_wo_sent,0.01,0.832443,l1,2000,constant,0.01,42


In [6]:
metrics_summary = results_to_df(results_par, ["metrics", "test", "aggregate"])
metrics_summary

,model,mae,mse,rmse,r2,directional_accuracy
0,linreg,0.021741,0.000960,0.030979,-0.054027,0.476764
1,linreg_wo_sent,0.021119,0.000937,0.030609,-0.029028,0.507745


In [7]:
models = sorted({r["kind"] for r in results if not r["kind"].endswith("_wo_sent")})

for model in models:
    subset = [r for r in results if r["kind"] == model]
    if not subset:
        continue

    print(f"\n── {model.upper()} ──")

    best_params = subset[0].get("best_params", {})
    if best_params:
        params_text = ", ".join(f"{k}={v}" for k, v in best_params.items())
        print(f"Params: {params_text}")

    metrics = subset[0]["metrics"]["test"]["aggregate"]
    line = " | ".join(f"{k.upper()}: {v:.4f}" for k, v in metrics.items())
    print(line)


── ENSEMBLE ──
Params: random_state=42
MAE: 0.0215 | MSE: 0.0009 | RMSE: 0.0304 | R2: -0.0123 | DIRECTIONAL_ACCURACY: 0.4888

── LINREG ──
Params: alpha=0.01, l1_ratio=0.8324426408004217, penalty=l1, max_iter=2000, learning_rate=constant, eta0=0.01, random_state=42
MAE: 0.0217 | MSE: 0.0010 | RMSE: 0.0310 | R2: -0.0540 | DIRECTIONAL_ACCURACY: 0.4768

── LSTM ──
Params: units=256, dense_units=128, dropout=0.028186489017586146, lr=0.0003624556684421663, weight_decay=2.0736933915042937e-07, epochs=500, batch_size=16, bidirectional=False, random_state=42
MAE: 0.0209 | MSE: 0.0010 | RMSE: 0.0309 | R2: -0.0497 | DIRECTIONAL_ACCURACY: 0.5508

── RANDOM_FOREST ──
Params: n_estimators=300, max_depth=8, min_samples_split=3, min_samples_leaf=5, max_features=log2, bootstrap=False, random_state=42
MAE: 0.0205 | MSE: 0.0009 | RMSE: 0.0293 | R2: 0.0557 | DIRECTIONAL_ACCURACY: 0.5680

── XGBOOST ──
Params: n_estimators=300, learning_rate=0.1, max_depth=5, min_child_weight=3.5246324340229482, reg_alph

In [8]:
metrics_summary = results_to_df(results, ["metrics", "test", "per_horizon"])
horizon_list = cfg.runtime.horizon_list
mapping = {i+1: h for i, h in enumerate(horizon_list)}
metrics_summary["per_horizon"] = metrics_summary["per_horizon"].astype(int).map(mapping)
metrics_summary = metrics_summary.rename(columns={"per_horizon": "horizon_days"})
metrics_summary = metrics_summary.sort_values(
    by=["horizon_days", "model"],
    ascending=[True, True]
).reset_index(drop=True)
metrics_summary

,model,horizon_days,target_std,mae,mse,rmse,r2,directional_accuracy
0,ensemble,1,0.010940,0.008247,0.000123,0.011083,-0.031553,0.528497
1,ensemble_wo_sent,1,0.010940,0.008402,0.000127,0.011256,-0.064108,0.502591
2,linreg,1,0.010940,0.008357,0.000125,0.011171,-0.048052,0.523316
3,linreg_wo_sent,1,0.010940,0.008188,0.000120,0.010943,-0.005626,0.518135
4,lstm,1,0.010940,0.008219,0.000121,0.010986,-0.013687,0.512953
5,lstm_wo_sent,1,0.010940,0.008223,0.000121,0.010987,-0.013765,0.497409
6,random_forest,1,0.010940,0.008152,0.000119,0.010897,0.002784,0.533679
7,random_forest_wo_sent,1,0.010940,0.008172,0.000119,0.010897,0.002799,0.528497
8,xgboost,1,0.010940,0.008163,0.000119,0.010908,0.000829,0.528497
9,xgboost_wo_sent,1,0.010940,0.008331,0.000138,0.011760,-0.161423,0.569948
